# LB 25.36727 | DQI readout windows on Atakan's bb4287e7

**Public leaderboard 25.36727** (submission of version 1 of this notebook, 2026-11-11). Questions, reviews and ideas are very welcome in the comments: the goal is to improve these circuits together, step by step.

**What is new here.** The current best public DQI, Atakan Aldemir's `bb4287e7` (from *QCC: Shared Products and
Controlled Refits*), sits on the `cd71c482` line of the team *i want to run gemma 9b*. That line never received two
blocks that the older `e9f54be3` line (Atakan Aldemir, *QCC: Terminal Readout and Phase Folds*; kept in Heliosli's
`9fcda0c1`) already had: **controlled-swap readout windows**. A window is 12 controlled swaps `CSWAP c a b` whose two
outputs are both X-measured right after. Each pair can be replaced by
`CX a b; HMR a -> m1; CZ c b if m1; HMR b -> m2; BIT_INVERT m1 if m2`, which applies the same phase
`(-1)^(m_a a' + m_b b')` with the same classical record (a' = c ? b : a, b' = c ? a : b), with no Toffoli.
There are four such windows (two regions of the decoder, each computed and uncomputed), so **48 fewer Toffolis per
shot**, every shot.

We found them by profiling 13 public DQI circuits region by region (the decoder's INFO_TAG regions) and asking where an
older circuit is cheaper than the newest line. Only those two regions qualified; a third candidate (Heliosli's
retained product in `inverse12 products`) depends on regions that `bb4287e7` has since rewritten, and transplanting
it alone breaks the phase (we measured ~1.7% phase errors), so it is not used.

**Result (official verifier, 102,400 shots):** mean Toffolis 394,116.9 vs 394,165.2 for `bb4287e7`, same 892 qubits.
The decoder line keeps its inherited rare failures (about 6e-6 per shot), so, like every file of this line, its
board price depends on the board's own draw: at zero failures the DQI cost is 373.56e9 (-0.0123% vs `bb4287e7`).

| Task | File | Authors (see credits) |
|---|---|---|
| invsqrt | `6c6a734d` | manwithacat (HW-25 join), gengsr, maoxin123 and the lineage they credit |
| DQI | **this notebook's transplant** on `bb4287e7` | Atakan Aldemir; team i want to run gemma 9b; Heliosli / e9f54be3 line |
| chemistry | maoxin123 v22 loaders on table `8afd4c67` | maoxin123; ozertuu's cache and the authors it credits |

In [ ]:
import sys, os, pathlib, zipfile, json, time, subprocess, re, lzma, shutil, hashlib, difflib, math
T0 = time.time()
def log(*a): print(f"[{(time.time()-T0)/60:5.1f} min]", *a, flush=True)
COMP = pathlib.Path("/kaggle/input/competitions/google-verifiable-quantum-circuit-compilation")
KM = COMP / "kickmix"; VER = COMP / "verifiers"
sys.path.insert(0, str(KM))
import kickmix as km
inp = pathlib.Path("/kaggle/input"); W = pathlib.Path("/tmp/w"); W.mkdir(exist_ok=True)
sha = lambda p: hashlib.sha256(pathlib.Path(p).read_bytes()).hexdigest()

src = {}
for p in inp.rglob("*.kmb"):
    for w in ("cd71c4822aa0", "9fcda0c1aeb3", "6c6a734dbcb6"):
        if p.name.startswith(w): src[w] = p
for z in inp.rglob("submission.zip"):
    s = str(z).lower()
    if "shared-products" in s:
        with zipfile.ZipFile(z) as zz: zz.extract("dqi.kmb", "/tmp/bb42")
        src["bb42"] = pathlib.Path("/tmp/bb42/dqi.kmb")
    if "4097-toffoli" in s:
        with zipfile.ZipFile(z) as zz:
            names = [n for n in zz.namelist() if n.startswith("dfthcsos/")]
            zz.extractall("/tmp/chem", members=names)
assert sha(src["bb42"]).startswith("bb4287e7"), sha(src["bb42"])
assert sha(src["6c6a734dbcb6"]).startswith("6c6a734d")
log({k: sha(v)[:8] for k, v in src.items()}, sorted(os.listdir("/tmp/chem/dfthcsos")))

## Build the DQI: replace two regions of `bb4287e7` by their `9fcda0c1` versions
The regions are identified by INFO_TAG order; the code asserts that `bb4287e7`'s region is byte-identical to
`cd71c482`'s (so the swap is exactly the cd71 -> e9f54be3 difference) and that the new regions introduce no new bits.

In [ ]:
for k in ("cd71c4822aa0", "9fcda0c1aeb3", "bb42"):
    km.Circuit.from_file(src[k], format="kmb").to_file(W / f"{k}.kmx", format="kmx")
bitre = re.compile(r"\bb(\d+)\b")
def scan(path, keep):
    tags, kept, idx = [""], {}, 0
    for line in open(path):
        if line.startswith(("APPEND_TO_REGISTER", "REGISTER")): continue
        if line.startswith("INFO_TAG"): idx += 1; tags.append(line)
        if idx in keep: kept.setdefault(idx, []).append(line)
    return tags, kept
TARGET = [673, 16783]
tagsA, A = scan(W / "cd71c4822aa0.kmx", set(TARGET))
tagsH, H = scan(W / "9fcda0c1aeb3.kmx", set(TARGET))
tagsB, _ = scan(W / "bb42.kmx", set())
sm = difflib.SequenceMatcher(a=tagsA, b=tagsB, autojunk=False)
amap = {blk.a + k: blk.b + k for blk in sm.get_matching_blocks() for k in range(blk.size)}
jmap = {amap[i]: i for i in TARGET}
_, Bk = scan(W / "bb42.kmx", set(jmap))
bits = lambda lines: {m for l in lines for m in bitre.findall(l)}
for j, i in jmap.items():
    assert Bk[j] == A[i] and not (bits(H[i]) - bits(A[i]))
    d = list(difflib.unified_diff(A[i], H[i], n=0))
    print(f"region {i}: {sum(l.startswith('-CSWAP') for l in d)} CSWAP removed, {len(d)} diff lines")
idx, done = 0, set()
with open("/tmp/dqi.kmx", "w") as f:
    for line in open(W / "bb42.kmx"):
        if line.startswith("INFO_TAG"): idx += 1
        if idx in jmap and not line.startswith(("APPEND_TO_REGISTER", "REGISTER")):
            if idx not in done: f.writelines(H[jmap[idx]]); done.add(idx)
            continue
        f.write(line)
pathlib.Path("/kaggle/working/build").mkdir(exist_ok=True)
km.Circuit.from_file("/tmp/dqi.kmx", format="kmx").to_file("/tmp/dqi.kmb", format="kmb")
log("dqi built", sha("/tmp/dqi.kmb")[:12])

## Assemble and verify with the unchanged official verifiers

In [ ]:
SUB = pathlib.Path("/tmp/sub"); (SUB / "dfthcsos").mkdir(parents=True, exist_ok=True)
shutil.copy(src["6c6a734dbcb6"], SUB / "invsqrt.kmb"); shutil.copy("/tmp/dqi.kmb", SUB / "dqi.kmb")
for p in pathlib.Path("/tmp/chem/dfthcsos").iterdir(): shutil.copy(p, SUB / "dfthcsos" / p.name)
def verify(args, timeout=4 * 3600):
    env = dict(os.environ, PYTHONPATH=f"{KM}:{VER}")
    r = subprocess.run([sys.executable, *map(str, args)], capture_output=True, text=True, env=env, cwd=str(VER), timeout=timeout)
    try: return json.loads(r.stdout[r.stdout.index("{"):])
    except Exception: return {"status": "ERROR", "stderr": r.stderr[-2000:]}
res = {}
res["dqi"] = verify([VER / "dqi/verify_submission_dqi.py", SUB / "dqi.kmb"]); log("dqi", {k: res["dqi"].get(k) for k in ("status", "score", "toffoli_count", "num_qubits", "value_errors", "phase_errors", "repetition_factor")})
res["dfthcsos"] = verify([VER / "dfthcsos/verify.py", SUB / "dfthcsos"]); log("chem", {k: res["dfthcsos"].get(k) for k in ("status", "score", "toffoli_count", "num_qubits", "lambda_eff")})
res["invsqrt"] = verify([VER / "invsqrt/verify_submission_invsqrt.py", SUB / "invsqrt.kmb"]); log("invsqrt", {k: res["invsqrt"].get(k) for k in ("status", "score", "num_qubits", "toffoli_count", "worst_error_over_tolerance")})

In [ ]:
REF = {"invsqrt": 2076774.23, "dqi": 24439463583672.93, "dfthcsos": 3355599588.32}
dqi_zero = res["dqi"]["score"] / res["dqi"]["repetition_factor"] * 1.1912604648115832  # price at zero failed shots
for name, dq in (("at this run's draw", res["dqi"]["score"]), ("at zero failed shots", dqi_zero)):
    r = {"invsqrt": REF["invsqrt"] / res["invsqrt"]["score"], "dqi": REF["dqi"] / dq, "dfthcsos": REF["dfthcsos"] / res["dfthcsos"]["score"]}
    print(f"{name:22s}", {k: round(v, 6) for k, v in r.items()}, "GM", round(math.prod(r.values()) ** (1 / 3), 5))
with zipfile.ZipFile("/kaggle/working/submission.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(SUB.rglob("*")):
        if p.is_file(): z.write(p, p.relative_to(SUB))
json.dump({"sha256": {str(p.relative_to(SUB)): sha(p) for p in SUB.rglob("*") if p.is_file()}, "verify": res}, open("/kaggle/working/receipt.json", "w"), indent=1, default=str)
log("submission.zip written")

## Credits and licences
- **DQI**: Atakan Aldemir (`bb4287e7`, *QCC: Shared Products and Controlled Refits*; and the controlled-swap readout
  windows of `e9f54be3`, *QCC: Terminal Readout and Phase Folds*); team *i want to run gemma 9b* (`b4dc000c`,
  `cd71c482`); Heliosli (`9fcda0c1`, the file the two regions are copied from) and the decoder line they credit
  (Syed Asad Ali, SomeJoint, jday96314, Kirill Tymchishin and others). Circuits taken from manwithacat's
  *VQCC public circuits* dataset (Apache-2.0) and Atakan Aldemir's notebook output.
- **invsqrt** `6c6a734d`: manwithacat (James Barlow, HW-25 join of gengsr's `26ebf39d` and maoxin123's `a15b0d73`),
  and the lineage they credit (Syed Asad Ali, Hosen42, Kirill Tymchishin, Atakan Aldemir, Ertuğrul Özer, busyaprime,
  yoyobar, mrmeows).
- **Chemistry**: maoxin123 (*QCC v22 | 4097-Toffoli chemistry loaders*), on ozertuu's table `8afd4c67` and the authors
  it credits (busyaprime, Heliosli, Lera/marculera, prvsiyan, Atakan Aldemir, Ertuğrul Özer, Syed Asad Ali, Ahmed
  Berat Özer).
- Tasks, kickmix and verifiers: Google Quantum AI / Qualtran.

Each circuit keeps its authors' licence (Apache-2.0 where stated). The code of this notebook is Apache-2.0.